# Class-distribution baseline for fMoW-WILDS

A model that only learns the *training* class distribution $p_i = n_i / N$ and predicts a label for every input by sampling from $p$ (ignoring the input) has expected accuracy

$$\text{acc} = \sum_i p_i \cdot q_i$$

where $q$ is the true class distribution of the evaluated split. This is the chance-agreement baseline (the $p_e$ term of Cohen's kappa; equals $1 - \text{Gini impurity}$ when $p=q$).

Here $p$ is fixed to the fMoW-WILDS **train** class distribution, and $q$ is swept over train / `id_test` / `test` (OOD) / per-region OOD, mirroring the ID / OOD / worst-region-accuracy (WRA) triple reported for the real models.

In [1]:
from wilds import get_dataset
import numpy as np
import pandas as pd

dataset = get_dataset(dataset="fmow", download=False, root_dir="/home/erik/git/bigpicture/data")
n_classes = dataset.n_classes
region_col = dataset.metadata_fields.index("region")
DOMAIN_NAMES = ["Asia", "Europe", "Africa", "Americas", "Oceania"]  # excludes "Other", same as the real WRA eval
region_name_to_id = {name: i for i, name in enumerate(dataset.metadata_map["region"])}

print("n_classes:", n_classes)
print("metadata_fields:", dataset.metadata_fields)
print("regions:", dataset.metadata_map["region"])

n_classes: 62
metadata_fields: ['region', 'year', 'y', 'from_source_domain']
regions: ['Asia', 'Europe', 'Africa', 'Americas', 'Oceania', 'Other']


In [2]:
def class_dist(y):
    """Empirical class distribution q_i = n_i / N."""
    counts = np.bincount(np.asarray(y), minlength=n_classes)
    return counts / counts.sum()


def split_y_region(split_name):
    indices = dataset.get_subset(split_name).indices
    y = dataset.y_array[indices].numpy()
    region = dataset.metadata_array[indices, region_col].numpy()
    return y, region


y_train, _ = split_y_region("train")
y_id_test, _ = split_y_region("id_test")
y_test, region_test = split_y_region("test")

p_train = class_dist(y_train)
print(f"train={len(y_train)}  id_test={len(y_id_test)}  test(ood)={len(y_test)}")

train=76863  id_test=11327  test(ood)=22108


In [3]:
train_acc = float(np.sum(p_train ** 2))
id_test_acc = float(np.sum(p_train * class_dist(y_id_test)))
ood_test_acc = float(np.sum(p_train * class_dist(y_test)))

region_accs = {}
for name in DOMAIN_NAMES:
    rid = region_name_to_id[name]
    mask = region_test == rid
    region_accs[name] = float(np.sum(p_train * class_dist(y_test[mask])))

ood_worst_region_name = min(region_accs, key=region_accs.get)
ood_worst_region_acc = region_accs[ood_worst_region_name]

print(f"train accuracy:              {train_acc:.4f}")
print(f"test-id accuracy:            {id_test_acc:.4f}")
print(f"test-ood accuracy:           {ood_test_acc:.4f}")
print(f"test-ood worst-region (WRA): {ood_worst_region_acc:.4f}  ({ood_worst_region_name})")

train accuracy:              0.0288
test-id accuracy:            0.0287
test-ood accuracy:           0.0271
test-ood worst-region (WRA): 0.0246  (Asia)


In [4]:
summary = pd.DataFrame({
    "split": ["train", "test-id", "test-ood", "test-ood-worst-region"],
    "accuracy": [train_acc, id_test_acc, ood_test_acc, ood_worst_region_acc],
})
print(summary.to_string(index=False))

print("\nPer-region test-ood accuracy:")
print(pd.Series(region_accs, name="accuracy").sort_values().to_string())

                split  accuracy
                train  0.028796
              test-id  0.028650
             test-ood  0.027064
test-ood-worst-region  0.024569

Per-region test-ood accuracy:
Asia        0.024569
Africa      0.025311
Europe      0.026278
Americas    0.029344
Oceania     0.032008


## Region-oracle baseline: region-conditional class distribution

Now suppose the model learns a *separate* class distribution per region from train, $p_{r,i} = n_{r,i}/n_r$, and at inference time correctly identifies the sample's region (an oracle) before sampling a label from that region's train distribution.

For an evaluated split $S$, partitioned into regions with weights $w_r^S = n_r^S / N^S$ and true per-region class distribution $q_r^S$, expected accuracy is the region-weighted average of the same chance-agreement term:

$$\text{acc}(S) = \sum_r w_r^S \sum_i p_{r,i}^{train} \cdot q_{r,i}^S$$

The pooled baseline above is the special case of treating the whole world as one region. Since region correlates with class here (land-use categories are not evenly spread across continents), conditioning on the oracle region should only help train/test-id/test-ood on average — but it can *shuffle which region is worst*, since regions improve by different amounts.

In [5]:
all_regions = dataset.metadata_map["region"]  # includes "Other"
_, region_train = split_y_region("train")
_, region_id_test = split_y_region("id_test")

p_region_train = {}
for name in all_regions:
    rid = region_name_to_id[name]
    mask = region_train == rid
    p_region_train[name] = class_dist(y_train[mask]) if mask.sum() > 0 else None
    print(f"{name:8s} train n={int(mask.sum())}")

Asia     train n=17809
Europe   train n=34816
Africa   train n=1582
Americas train n=20973
Oceania  train n=1641
Other    train n=42


In [6]:
def region_oracle_accuracy(y, region):
    """Region-weighted expected accuracy: oracle region id + region-conditional train class dist."""
    total = len(y)
    acc = 0.0
    per_region = {}
    for rid in np.unique(region):
        name = all_regions[rid]
        if p_region_train.get(name) is None:
            continue
        mask = region == rid
        n_r = int(mask.sum())
        region_acc = float(np.sum(p_region_train[name] * class_dist(y[mask])))
        per_region[name] = region_acc
        acc += (n_r / total) * region_acc
    return acc, per_region


train_acc_oracle, _ = region_oracle_accuracy(y_train, region_train)
id_test_acc_oracle, _ = region_oracle_accuracy(y_id_test, region_id_test)
ood_test_acc_oracle, region_accs_oracle = region_oracle_accuracy(y_test, region_test)

ood_worst_region_name_oracle = min(DOMAIN_NAMES, key=lambda n: region_accs_oracle[n])
ood_worst_region_acc_oracle = region_accs_oracle[ood_worst_region_name_oracle]

print(f"train accuracy:              {train_acc_oracle:.4f}")
print(f"test-id accuracy:            {id_test_acc_oracle:.4f}")
print(f"test-ood accuracy:           {ood_test_acc_oracle:.4f}")
print(f"test-ood worst-region (WRA): {ood_worst_region_acc_oracle:.4f}  ({ood_worst_region_name_oracle})")

train accuracy:              0.0414
test-id accuracy:            0.0414
test-ood accuracy:           0.0395
test-ood worst-region (WRA): 0.0317  (Europe)


In [7]:
comparison = pd.DataFrame({
    "split": ["train", "test-id", "test-ood", "test-ood-worst-region"],
    "pooled_train_dist": [train_acc, id_test_acc, ood_test_acc, ood_worst_region_acc],
    "region_oracle_dist": [train_acc_oracle, id_test_acc_oracle, ood_test_acc_oracle, ood_worst_region_acc_oracle],
})
print(comparison.to_string(index=False))

print("\nPer-region test-ood accuracy, pooled vs. region-oracle:")
region_comparison = pd.DataFrame({
    "pooled": pd.Series(region_accs),
    "region_oracle": pd.Series({n: region_accs_oracle[n] for n in DOMAIN_NAMES}),
}).sort_values("region_oracle")
print(region_comparison.to_string())

                split  pooled_train_dist  region_oracle_dist
                train           0.028796            0.041369
              test-id           0.028650            0.041428
             test-ood           0.027064            0.039522
test-ood-worst-region           0.024569            0.031670

Per-region test-ood accuracy, pooled vs. region-oracle:
            pooled  region_oracle
Europe    0.026278       0.031670
Asia      0.024569       0.039814
Americas  0.029344       0.041035
Oceania   0.032008       0.041036
Africa    0.025311       0.051629


## Why can a real location encoder reach ~25pp on test-id?

5 discrete continents is a very coarse partition. A location encoder gets *continuous* (lat, lon) and can learn $p(y \mid \text{location})$ at whatever granularity the data supports. If accuracy keeps rising as the geographic partition gets finer (plausible: many fMoW categories, e.g. `nuclear_powerplant`, `space_facility`, `border_checkpoint`, are extremely location-specific), continent (5 groups) is just one coarse point on that curve. Check the next point: country (146 unique codes in train).

In [8]:
def split_y_country(split_name):
    indices = dataset.get_subset(split_name).indices
    y = dataset.y_array[indices].numpy()
    orig_idx = dataset.full_idxs[indices]
    country = dataset.metadata["country_code"].values[orig_idx]
    return y, country


y_train_c, country_train = split_y_country("train")
y_id_test_c, country_id_test = split_y_country("id_test")
y_test_c, country_test = split_y_country("test")

p_country_train = {
    c: class_dist(y_train_c[country_train == c])
    for c in set(country_train)
}
print("unique countries: train=%d  id_test=%d  test(ood)=%d"
      % (len(p_country_train), len(set(country_id_test)), len(set(country_test))))
print("countries in id_test unseen in train:", set(country_id_test) - set(p_country_train))


def country_oracle_accuracy(y, country):
    total = len(y)
    acc = 0.0
    for c in set(country):
        if c not in p_country_train:
            continue  # unseen country -> no train-side distribution to condition on
        mask = country == c
        acc += (mask.sum() / total) * float(np.sum(p_country_train[c] * class_dist(y[mask])))
    return acc


train_acc_country = country_oracle_accuracy(y_train_c, country_train)
id_test_acc_country = country_oracle_accuracy(y_id_test_c, country_id_test)
ood_test_acc_country = country_oracle_accuracy(y_test_c, country_test)

print(f"train accuracy:    {train_acc_country:.4f}")
print(f"test-id accuracy:  {id_test_acc_country:.4f}")
print(f"test-ood accuracy: {ood_test_acc_country:.4f}")

unique countries: train=146  id_test=113  test(ood)=171
countries in id_test unseen in train: set()


train accuracy:    0.1030
test-id accuracy:  0.1032
test-ood accuracy: 0.0803


Country-level conditioning alone triples the region-oracle number on test-id (~4.1% → ~10.3%). That confirms the granularity story but doesn't close the gap to 25pp — a trained location encoder isn't limited to 146 discrete country buckets, it fits a continuous, nonlinear $p(y \mid \text{lat, lon})$ that can separate cities, coastlines, or facility clusters *within* a country.

One alternative (non-generalization) explanation worth ruling out: fMoW re-images the same physical site over time (same lat/lon, several timestamps). If such a site's repeats were split across `train` and `id_test`, a location encoder could partly be doing lookup rather than learning a geographic prior. Check how often that happens.

In [9]:
raw = pd.read_csv(
    "/home/erik/git/bigpicture/data/fmow_v1.1/rgb_metadata.csv",
    usecols=["split", "lat", "lon"],
)
# raw['split'] is the *official* fMoW split (train/val/test/seq), pre-dating the
# WILDS time-based ID/OOD re-split, but train vs. id_test never mix raw splits
# (id_test is drawn from raw 'test' rows), so exact-coordinate overlap here upper-bounds
# how much train/id_test site lookup could be happening.
n_multi_split_coords = (raw.groupby(["lat", "lon"])["split"].nunique() > 1).sum()
n_coords = raw.groupby(["lat", "lon"]).ngroups
print(f"lat/lon groups spanning >1 raw split: {n_multi_split_coords} / {n_coords} "
      f"({100 * n_multi_split_coords / n_coords:.2f}%)")

lat/lon groups spanning >1 raw split: 98 / 203255 (0.05%)


Only a tiny fraction of exact coordinates span more than one raw split, so this rules out "the encoder is mostly looking up repeated sites" as the main story — the effect is real but small.

**Conclusion:** the ladder pooled (2.9%) → 5-region oracle (4.1%) → 146-country oracle (10.3%) is already a 3.5x rise from adding geographic granularity alone, all with hard discrete buckets and no learned interpolation. A trained location encoder (a) partitions space continuously rather than by administrative border, so it can separate cities/clusters *within* a country, and (b) is optimized end-to-end for class-discriminative geography rather than matched to country boundaries by construction. 25pp on test-id is a large number in isolation, but it is the expected continuation of a trend this notebook already shows to be strongly monotone in geographic granularity — not evidence that the location encoder is doing something outside what location alone should be able to predict.